# 🐘 HDFS CRUD 실습 (WebHDFS + Python)

JupyterLab 컨테이너에서 Hadoop NameNode에 접속하여 HDFS 파일 시스템을 조작합니다.

| 항목 | 값 |
|------|----|
| NameNode 주소 | `http://namenode:9870` |
| 접속 방식 | WebHDFS REST API (`hdfs` 라이브러리) |
| 실습 내용 | Create / Read / Update / Delete |

---
## 0. 환경 설정

In [ ]:
# !pip install hdfs pandas

In [1]:
# 필요한 라이브러리 import
from hdfs import InsecureClient
import pandas as pd
import io

# HDFS 클라이언트 생성 (WebHDFS 방식)
# - namenode: Hadoop NameNode의 호스트명 (Docker 네트워크 내에서 해석)
# - 9870: WebHDFS REST API 포트
# - user='root': HDFS 접근 사용자
client = InsecureClient('http://namenode:9870', user='root')

print('✅ HDFS 클라이언트 생성 완료')
print(f'   접속 주소: http://namenode:9870')
print(f'   사용자: root')

✅ HDFS 클라이언트 생성 완료
   접속 주소: http://namenode:9870
   사용자: root


---
## 1. 연결 확인 (Read)

HDFS 루트(`/`) 디렉토리 목록을 조회하여 연결이 정상인지 확인합니다.

In [2]:
# HDFS 루트 디렉토리 목록 조회
root_contents = client.list('/')

print('📁 HDFS 루트(/) 디렉토리 목록:')
print('=' * 40)
for item in root_contents:
    print(f'  📂 /{item}')
print('=' * 40)
print(f'\n✅ 연결 성공! 총 {len(root_contents)}개 항목 확인')

📁 HDFS 루트(/) 디렉토리 목록:
  📂 /ref
  📂 /test

✅ 연결 성공! 총 2개 항목 확인


---
## 2. 디렉토리 생성 (Create)

HDFS에 새로운 디렉토리를 생성합니다. `makedirs()`는 중간 경로도 함께 생성합니다.

In [3]:
# 작업용 디렉토리 생성
dir_path = '/data/pknu'

# makedirs: 중간 디렉토리까지 한번에 생성 (mkdir -p 와 동일)
client.makedirs(dir_path)

print(f'📂 디렉토리 생성 완료: {dir_path}')

# 생성 확인
status = client.status(dir_path)
print(f'\n📋 디렉토리 상태 정보:')
print(f'   타입: {status["type"]}')
print(f'   소유자: {status["owner"]}')
print(f'   권한: {status["permission"]}')

📂 디렉토리 생성 완료: /data/pknu

📋 디렉토리 상태 정보:
   타입: DIRECTORY
   소유자: root
   권한: 755


---
## 3. 파일 업로드 - Create

`write()` 메서드로 Python 데이터를 HDFS에 파일로 저장합니다.

In [4]:
# (1) 텍스트 파일 업로드
dir_path = '/data/pknu/'
txt_path = dir_path + 'hello.txt'
txt_content = '안녕하세요! Hadoop HDFS 실습입니다.\n이 파일은 JupyterLab에서 WebHDFS로 업로드되었습니다.\n'

with client.write(txt_path, encoding='utf-8', overwrite=True) as f:
    f.write(txt_content)

print(f'📄 텍스트 파일 업로드 완료: {txt_path}')

# (2) CSV 파일 업로드
csv_path = dir_path + '/students.csv'
csv_content = """이름,학번,학과,성적
김철수,2024001,컴퓨터공학,95
이영희,2024002,데이터과학,88
박민수,2024003,인공지능,92
정수진,2024004,컴퓨터공학,85
한지우,2024005,데이터과학,97
"""

with client.write(csv_path, encoding='utf-8', overwrite=True) as f:
    f.write(csv_content)

print(f'📄 CSV 파일 업로드 완료: {csv_path}')
print(f'\n✅ 총 2개 파일 업로드 성공!')

📄 텍스트 파일 업로드 완료: /data/pknu/hello.txt
📄 CSV 파일 업로드 완료: /data/pknu//students.csv

✅ 총 2개 파일 업로드 성공!


---
## 4. 파일 목록 조회 - Read (list / status)

디렉토리 내 파일 리스트와 각 파일의 상세 정보를 조회합니다.

In [5]:
# 디렉토리 내 파일 목록 조회
dir_path = '/data/pknu/'
file_list = client.list(dir_path, status=True)

print(f'📁 {dir_path} 디렉토리 내용:')
print('=' * 70)
print(f'{"파일명":<25} {"타입":<10} {"크기(bytes)":<15} {"소유자":<10}')
print('-' * 70)

for name, info in file_list:
    file_type = '📂 DIR' if info['type'] == 'DIRECTORY' else '📄 FILE'
    print(f'{name:<25} {file_type:<10} {info["length"]:<15} {info["owner"]:<10}')

print('=' * 70)
print(f'총 {len(file_list)}개 항목')

📁 /data/pknu/ 디렉토리 내용:
파일명                       타입         크기(bytes)       소유자       
----------------------------------------------------------------------
hello.txt                 📄 FILE     114             root      
students.csv              📄 FILE     210             root      
총 2개 항목


In [6]:
# 특정 파일의 상세 정보 조회
dir_path = '/data/pknu/'
file_path = dir_path + 'students.csv'
file_status = client.status(file_path)

print(f'📋 파일 상세 정보: {file_path}')
print('=' * 50)
for key, value in file_status.items():
    print(f'  {key}: {value}')

📋 파일 상세 정보: /data/pknu/students.csv
  accessTime: 1779345290655
  blockSize: 134217728
  childrenNum: 0
  fileId: 16421
  group: supergroup
  length: 210
  modificationTime: 1779345290674
  owner: root
  pathSuffix: 
  permission: 644
  replication: 3
  storagePolicy: 0
  type: FILE


---
## 5. 파일 내용 읽기 - Read

HDFS에 저장된 파일의 내용을 Python으로 읽어옵니다.

In [7]:
# (1) 텍스트 파일 읽기
dir_path = '/data/pknu/'
txt_path = dir_path + 'hello.txt'

with client.read(txt_path, encoding='utf-8') as reader:
    content = reader.read()

print(f'📄 {txt_path} 내용:')
print('=' * 50)
print(content)
print('=' * 50)

📄 /data/pknu/hello.txt 내용:
안녕하세요! Hadoop HDFS 실습입니다.
이 파일은 JupyterLab에서 WebHDFS로 업로드되었습니다.



In [8]:
# (2) CSV 파일을 pandas DataFrame으로 읽기
dir_path = '/data/pknu/'
csv_path = dir_path + 'students.csv'

with client.read(csv_path, encoding='utf-8') as reader:
    csv_data = reader.read()

df = pd.read_csv(io.StringIO(csv_data))

print(f'📊 {csv_path} → pandas DataFrame:')
print()
df

📊 /data/pknu/students.csv → pandas DataFrame:



,이름,학번,학과,성적
0,김철수,2024001,컴퓨터공학,95
1,이영희,2024002,데이터과학,88
2,박민수,2024003,인공지능,92
3,정수진,2024004,컴퓨터공학,85
4,한지우,2024005,데이터과학,97


---
## 6. 파일 수정 - Update

HDFS는 기본적으로 **append-only** 파일 시스템입니다.
- **덮어쓰기(overwrite)**: `write(overwrite=True)`로 파일 전체를 교체
- **이어쓰기(append)**: 기존 파일 뒤에 데이터 추가 (Hadoop 설정 필요)

In [9]:
# (1) 텍스트 파일 덮어쓰기 (Update)
dir_path = '/data/pknu/'
txt_path = dir_path + 'hello.txt'

# 수정 전 내용 확인
with client.read(txt_path, encoding='utf-8') as reader:
    before = reader.read()
print('📝 수정 전:')
print(before)

# 새 내용으로 덮어쓰기
new_content = '파일이 수정되었습니다! (Update 완료)\n수정 시각: JupyterLab에서 WebHDFS로 갱신\n빅데이터 실습 화이팅! 🎉\n'

with client.write(txt_path, encoding='utf-8', overwrite=True) as writer:
    writer.write(new_content)

# 수정 후 내용 확인
with client.read(txt_path, encoding='utf-8') as reader:
    after = reader.read()
print('📝 수정 후:')
print(after)
print('✅ 파일 수정(Update) 완료!')

📝 수정 전:
안녕하세요! Hadoop HDFS 실습입니다.
이 파일은 JupyterLab에서 WebHDFS로 업로드되었습니다.

📝 수정 후:
파일이 수정되었습니다! (Update 완료)
수정 시각: JupyterLab에서 WebHDFS로 갱신
빅데이터 실습 화이팅! 🎉

✅ 파일 수정(Update) 완료!


In [10]:
# (2) CSV 파일에 데이터 추가 (덮어쓰기 방식)
dir_path = '/data/pknu/'
csv_path = dir_path + 'students.csv'

# 기존 데이터 읽기
with client.read(csv_path, encoding='utf-8') as reader:
    csv_data = reader.read()

df = pd.read_csv(io.StringIO(csv_data))
print(f'📊 수정 전: {len(df)}명')
print(df)

# 새 학생 데이터 추가
new_students = pd.DataFrame([
    {'이름': '최우영', '학번': 2024006, '학과': '인공지능', '성적': 90},
    {'이름': '강예린', '학번': 2024007, '학과': '컴퓨터공학', '성적': 93},
])

df = pd.concat([df, new_students], ignore_index=True)

# 수정된 DataFrame을 HDFS에 저장 (덮어쓰기)
with client.write(csv_path, encoding='utf-8', overwrite=True) as writer:
    df.to_csv(writer, index=False)

print(f'\n📊 수정 후: {len(df)}명')
print(df)
print('\n✅ CSV 데이터 추가(Update) 완료!')

📊 수정 전: 5명
    이름       학번     학과  성적
0  김철수  2024001  컴퓨터공학  95
1  이영희  2024002  데이터과학  88
2  박민수  2024003   인공지능  92
3  정수진  2024004  컴퓨터공학  85
4  한지우  2024005  데이터과학  97

📊 수정 후: 7명
    이름       학번     학과  성적
0  김철수  2024001  컴퓨터공학  95
1  이영희  2024002  데이터과학  88
2  박민수  2024003   인공지능  92
3  정수진  2024004  컴퓨터공학  85
4  한지우  2024005  데이터과학  97
5  최우영  2024006   인공지능  90
6  강예린  2024007  컴퓨터공학  93

✅ CSV 데이터 추가(Update) 완료!


---
## 7. 파일/디렉토리 삭제 - Delete

`delete()` 메서드로 HDFS의 파일이나 디렉토리를 삭제합니다.
- `recursive=True`: 디렉토리 내 모든 내용을 함께 삭제

In [11]:
# (1) 개별 파일 삭제
dir_path = '/data/pknu/'
txt_path = dir_path + 'hello.txt'

# 삭제 전 확인
print('📁 삭제 전 파일 목록:')
for name in client.list(dir_path):
    print(f'  📄 {name}')

# 파일 삭제
result = client.delete(txt_path)
print(f'\n🗑️ {txt_path} 삭제 결과: {result}')

# 삭제 후 확인
print('\n📁 삭제 후 파일 목록:')
for name in client.list(dir_path):
    print(f'  📄 {name}')

📁 삭제 전 파일 목록:
  📄 hello.txt
  📄 students.csv

🗑️ /data/pknu/hello.txt 삭제 결과: True

📁 삭제 후 파일 목록:
  📄 students.csv


In [12]:
# (2) 디렉토리 전체 삭제 (recursive=True)
dir_path = '/data'
del_dir = dir_path+ '/pknu'
# 삭제 전 확인
print('📁 삭제 전 /pknu 디렉토리:')
for name in client.list(del_dir):
    print(f'  📂 {name}')

# 디렉토리 및 하위 내용 모두 삭제
result = client.delete(del_dir, recursive=True)
print(f'\n🗑️ {dir_path} 삭제 결과: {result}')

# 삭제 후 확인
print('\n📁 삭제 후 /data 디렉토리:')
remaining = client.list(dir_path )
if remaining:
    for name in remaining:
        print(f'  📂 {name}')
else:
    print('  (비어 있음)')

print('\n✅ 디렉토리 삭제(Delete) 완료!')

📁 삭제 전 /pknu 디렉토리:
  📂 students.csv

🗑️ /data 삭제 결과: True

📁 삭제 후 /data 디렉토리:
  (비어 있음)

✅ 디렉토리 삭제(Delete) 완료!


---
## 8. 종합 실습: CSV 데이터 파이프라인

전체 CRUD 흐름을 하나로 묶어 실습합니다:

**Create** → 디렉토리 생성 + CSV 업로드  
**Read** → HDFS에서 읽어 DataFrame 변환  
**Update** → 데이터 가공 후 재저장  
**Delete** → 정리

In [13]:
# ========================================
# 📌 CREATE: 디렉토리 생성 + 매출 데이터 업로드
# ========================================
work_dir = '/data/sales'
client.makedirs(work_dir)

sales_csv = """날짜,상품,수량,단가
2024-01-15,노트북,3,1200000
2024-01-15,마우스,10,25000
2024-01-16,키보드,5,75000
2024-01-16,모니터,2,350000
2024-01-17,노트북,1,1200000
2024-01-17,마우스,8,25000
2024-01-17,키보드,3,75000
"""

sales_path = f'{work_dir}/sales_raw.csv'
with client.write(sales_path, encoding='utf-8', overwrite=True) as writer:
    writer.write(sales_csv)

print(f'✅ CREATE 완료: {sales_path}')

✅ CREATE 완료: /data/sales/sales_raw.csv


In [14]:
# ========================================
# 📌 READ: HDFS에서 읽어 DataFrame으로 변환
# ========================================
with client.read(sales_path, encoding='utf-8') as reader:
    raw_data = reader.read()

df_sales = pd.read_csv(io.StringIO(raw_data))

print('📊 원본 매출 데이터 (HDFS에서 읽기):')
print()
df_sales

📊 원본 매출 데이터 (HDFS에서 읽기):



,날짜,상품,수량,단가
0,2024-01-15,노트북,3,1200000
1,2024-01-15,마우스,10,25000
2,2024-01-16,키보드,5,75000
3,2024-01-16,모니터,2,350000
4,2024-01-17,노트북,1,1200000
5,2024-01-17,마우스,8,25000
6,2024-01-17,키보드,3,75000


In [15]:
# ========================================
# 📌 UPDATE: 데이터 가공 후 재저장
# ========================================

# 매출액 컬럼 추가 (수량 × 단가)
df_sales['매출액'] = df_sales['수량'] * df_sales['단가']

# 가공된 데이터를 새 파일로 저장
processed_path = f'{work_dir}/sales_processed.csv'
with client.write(processed_path, encoding='utf-8', overwrite=True) as writer:
    df_sales.to_csv(writer, index=False)

print('📊 가공된 매출 데이터 (매출액 컬럼 추가):')
print()
print(df_sales.to_string(index=False))

# 상품별 매출 요약
print('\n📈 상품별 매출 요약:')
summary = df_sales.groupby('상품')['매출액'].agg(['sum', 'count']).reset_index()
summary.columns = ['상품', '총매출액', '거래건수']
summary = summary.sort_values('총매출액', ascending=False)
print()
print(summary.to_string(index=False))

# 요약 데이터도 HDFS에 저장
summary_path = f'{work_dir}/sales_summary.csv'
with client.write(summary_path, encoding='utf-8', overwrite=True) as writer:
    summary.to_csv(writer, index=False)

print(f'\n✅ UPDATE 완료: {processed_path}, {summary_path}')

📊 가공된 매출 데이터 (매출액 컬럼 추가):

        날짜  상품  수량      단가     매출액
2024-01-15 노트북   3 1200000 3600000
2024-01-15 마우스  10   25000  250000
2024-01-16 키보드   5   75000  375000
2024-01-16 모니터   2  350000  700000
2024-01-17 노트북   1 1200000 1200000
2024-01-17 마우스   8   25000  200000
2024-01-17 키보드   3   75000  225000

📈 상품별 매출 요약:

 상품    총매출액  거래건수
노트북 4800000     2
모니터  700000     1
키보드  600000     2
마우스  450000     2

✅ UPDATE 완료: /data/sales/sales_processed.csv, /data/sales/sales_summary.csv


In [16]:
# ========================================
# 📌 최종 확인: HDFS 파일 목록
# ========================================
print(f'📁 {work_dir} 최종 파일 목록:')
print('=' * 60)

for name, info in client.list(work_dir, status=True):
    size_kb = info['length'] / 1024
    print(f'  📄 {name:<30} ({size_kb:.1f} KB)')

print('=' * 60)

📁 /data/sales 최종 파일 목록:
  📄 sales_processed.csv            (0.3 KB)
  📄 sales_raw.csv                  (0.2 KB)
  📄 sales_summary.csv              (0.1 KB)


In [17]:
# ========================================
# 📌 DELETE: 실습 데이터 정리
# ========================================
client.delete(work_dir, recursive=True)

print(f'🗑️ {work_dir} 디렉토리 및 모든 파일 삭제 완료')
print('\n🎉 종합 실습 (Create → Read → Update → Delete) 완료!')

🗑️ /data/sales 디렉토리 및 모든 파일 삭제 완료

🎉 종합 실습 (Create → Read → Update → Delete) 완료!
